In [45]:
from transformers import AutoModelForSequenceClassification, AutoConfig, AutoTokenizer, AdamW
import json
import torch
from tqdm import tqdm
import os
import matplotlib.pyplot as plt
import gc
import numpy as np
np.random.seed(42)
from scipy.stats import spearmanr
import openai
import time
import requests
import re
import pickle
from scipy.stats import spearmanr

from scipy.stats import spearmanr

# Convert to ranks
def calc_spearmanr(ytrue, ypredict):
    '''
    ytrue    = [1, 2, 3, 4, 5]
    ypredict = [1, 2, 4, 3, 4]
    '''
    ytrue_ranks = [ytrue.index(x) for x in ytrue]
    ypredict_ranks = [ytrue.index(x) for x in ypredict]
    rho, p_value = spearmanr(ytrue_ranks, ypredict_ranks)
    # print(f"Spearman's Rank Correlation: {rho:.4f}, p-value: {p_value:.4f}")
    return rho, p_value


def read_pkl(fname):
    with open(fname, 'rb') as fo:
        pkl_data = pickle.load(fo, encoding='bytes')
    return pkl_data

def write_pkl(pkl_data, fname):
    fo = open(fname, 'wb')
    pickle.dump(pkl_data, fo)
    print("pkl_file write over!")

def _load_data(data_path: str) -> None:
    
    """
    Load data from data_path.
    
    data_path: str - path to json data file.
    """
    
    try:
        with open(data_path, "r") as f:
            data = json.load(f)
    except json.JSONDecodeError:
        with open(data_path, "r") as f:
            data = [json.loads(line) for line in f]
    
    return data



openai.api_key = "YOUR_OPENAI_KEY"
device = torch.device("cuda:2")
rank_model='deberta-v3-base'
# stage 0 score model
optimize_stage = 4
complexity_model_name = f'../checkpoints/{rank_model}/alpaca-train-complexity-2k/_stage_{optimize_stage}/rank_net-1024-ylabel'
quality_model_name = f'../checkpoints/{rank_model}/alpaca-train-quality-2k/_stage_{optimize_stage}/rank_net-2048-ylabel'
tokenizer_path = f'../pre_trained_model/{rank_model}'

data_path = f'../data/deita_sota_pool/model_metric/_stage_0/qck_llama2-13b-hf_deita_sota_pool_305263_token_4096.json'
data_name = 'deita_sota_pool_305263_token_4096'
model_name = complexity_model_name

print(model_name)

../checkpoints/deberta-v3-base/alpaca-train-complexity-2k/_stage_4/rank_net-1024-ylabel


In [33]:
ypredict = [1, 2, 5, 4, 3]
ytrue    = [1, 2, 3, 4, 5]
calc_spearmanr(ytrue, ypredict)

(0.6, 0.28475697986529375)

In [34]:
model_name

'../checkpoints/deberta-v3-base/alpaca-train-complexity-2k/_stage_4/rank_net-1024-ylabel'

In [35]:
print(f'data_path={data_path}')
data = _load_data(data_path)
print(f'data[0]={data[0]}')

data_path=../data/deita_sota_pool/model_metric/_stage_0/qck_llama2-13b-hf_deita_sota_pool_305263_token_4096.json
data[0]={'id': 'LbHON3Q', 'source': 'ShareGPT', 'conversations': [{'from': 'human', 'value': 'People may now think of queries and replies in novel and creative ways thanks to new AI text systems. What are some crucial inquiries we may pose to these text-based AIs that would yield original and admirable insights into humanity? Make six specific questions that don’t contain the phrase “AI.”'}, {'from': 'gpt', 'value': '1. What is the meaning of life and how does it differ from individual to individual?\n2. How does our understanding of the world shape our beliefs and values?\n3. What are the most important qualities that make us human?\n4. How do our emotions influence our decision-making processes and behavior?\n5. What is the relationship between our thoughts and actions, and how does it impact our daily lives?\n6. Can humans find happiness and fulfillment by pursuing materi

In [5]:

'''
batch complexity single
'''
test_index = 0
config = AutoConfig.from_pretrained(complexity_model_name)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(complexity_model_name, config=config)


# complexity_data = _load_data('../data/RankGPT/stage_0/alpaca-train-complexity-2k.json')
# query = complexity_data[test_index]['query']
# batch = tokenizer(query, padding=True, truncation=True, return_tensors="pt", max_length=2048)
# out = model(**batch)
# out.logits.squeeze().tolist()

query = [conv['value'] for conv in data[0]['conversations'][::2]]
response = [conv['value'] for conv in data[0]['conversations'][1::2]]
batch = tokenizer(query, padding=True, truncation=True, return_tensors="pt", max_length=1024)
batch = {key: value for key, value in batch.items()}
out = model(**batch)
out.logits.squeeze().tolist()



/home/pclv100s-1/.conda/envs/llama_factory/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:473: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


[-2.4874348640441895, -8.55833625793457, -8.558334350585938]

In [13]:

print(device)
config = AutoConfig.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)
model = model.to(device)
# model = model.cuda()


model.enable_input_require_grads()
model.gradient_checkpointing_enable()

complexity_convs_score = []
for index,item in tqdm(enumerate(data)):

    human_conversations = [conv['value'] for conv in item['conversations'][::2]]
    gpt_conversations   = [conv['value'] for conv in item['conversations'][1::2]]
    
    if len(human_conversations) == len(gpt_conversations):
        conv_score = []
        for i in range(len(human_conversations)):
            batch = tokenizer(human_conversations[i], padding=True, truncation=True, return_tensors="pt", max_length=1024)
            # batch = {key: value.to(torch.device("cuda")) for key, value in batch.items()}
            batch = {key: value.to(device) for key, value in batch.items()}
            try:
                out = model(**batch)
            except:
                print('GC:\t', index)
                gc.collect()
                torch.cuda.empty_cache()
                out = model(**batch)
            conv_score.append(out.logits.squeeze().tolist()) # out.logits.squeeze().tolist()=[2.9446518421173096, 10.687034606933594, 9.427430152893066]

        complexity_convs_score.append(conv_score)
    else:
        print('error line:\t', index)

cuda:2


305263it [9:07:12,  9.30it/s]


In [9]:
complexity_convs_score

[[-5.588554382324219, -19.217561721801758, -19.217561721801758],
 [3.395634889602661],
 [-11.556612014770508],
 [10.460641860961914],
 [-17.294790267944336,
  -10.427579879760742,
  -8.52461051940918,
  -2.0347542762756348],
 [10.535307884216309],
 [-1.143235683441162, -19.706829071044922],
 [-4.15977668762207,
  -9.130966186523438,
  -12.132499694824219,
  -8.991962432861328,
  -12.33328628540039,
  -7.414058685302734],
 [14.937445640563965],
 [-7.53479528427124],
 [-5.412352561950684],
 [-13.598690032958984],
 [10.769657135009766],
 [7.4403276443481445],
 [13.172370910644531],
 [8.527356147766113, -4.32921028137207]]

In [10]:
complexity_model_name + f'/complexity_stage_{optimize_stage}_{data_name}.pkl'

'../checkpoints/deberta-v3-large/alpaca-train-complexity-2k/_stage_3_deberta-v3-large/rank_net-1024-ylabel/complexity_stage_3_deita_sota_pool_305263_token_4096.pkl'

In [36]:
complexity_model_name

'../checkpoints/deberta-v3-base/alpaca-train-complexity-2k/_stage_4/rank_net-1024-ylabel'

In [37]:




# write_pkl(complexity_convs_score, complexity_model_name + f'/complexity_stage_{optimize_stage}_{data_name}.pkl')

# quality_convs_score    = read_pkl(f'../checkpoints/{rank_model}/alpaca-train-quality-2k/_stage_{optimize_stage}_{rank_model}/rank_net-2048-ylabel/quality_stage_{optimize_stage}_deita_sota_pool_305263_token_4096.pkl')

quality_convs_score    = read_pkl(f'../checkpoints/{rank_model}/alpaca-train-quality-2k/_stage_3/rank_net-2048-ylabel/quality_stage_3_deita_sota_pool_305263_token_4096.pkl')

complexity_convs_score = read_pkl(complexity_model_name + f'/complexity_stage_{optimize_stage}_{data_name}.pkl')

In [38]:
len(quality_convs_score)

305263

In [39]:
print(quality_convs_score[:12])
print(complexity_convs_score[:12])

[[1.8894349336624146, 3.575186252593994, 4.76503324508667], [5.155736446380615], [5.400453090667725], [6.094990253448486], [2.436004877090454, 5.37188196182251, 6.270315170288086, 6.2935662269592285], [6.6326165199279785], [3.542778968811035, 3.848637104034424], [5.125317096710205, 5.3949174880981445, 6.743919849395752, 6.008814811706543, 6.240902423858643, 3.84055233001709], [7.72443151473999], [4.142751693725586], [4.557082653045654], [-5.952919960021973]]
[[1.0892025232315063, -11.041285514831543, -11.041285514831543], [1.4755693674087524], [-6.461551189422607], [2.366058588027954], [-5.642007827758789, -2.2246785163879395, 0.4768742620944977, -0.8615173697471619], [4.628573417663574], [0.6233891844749451, -5.5893473625183105], [0.37804996967315674, 0.5434896349906921, -2.8524510860443115, 0.06409231573343277, -1.3266485929489136, -5.570372104644775], [2.9875402450561523], [-3.8298776149749756], [-3.026639938354492], [-2.8799493312835693]]


In [40]:
len(complexity_convs_score)

305263

In [41]:
add_complexity_quality_scores = []
for index,item in enumerate(data):
    item['complexity_scores'] = complexity_convs_score[index]
    item['quality_scores'] = quality_convs_score[index]
    add_complexity_quality_scores.append(item)



add_complexity_quality_scores_filename = '/'.join(data_path.split('/')[:-2]) + f'/_stage_{optimize_stage}_{rank_model}/' + data_path.split('/')[-1]
print(add_complexity_quality_scores_filename)


../data/deita_sota_pool/model_metric/_stage_4_deberta-v3-base/qck_llama2-13b-hf_deita_sota_pool_305263_token_4096.json


In [42]:

model_name_list = ['llama2-13b-hf', 'Mistral-7B-v0.1', 'llama-13b-hf'] # 'llama-7b-hf', 'llama2-7b-hf'

for model_name in model_name_list:
    data_dir = '../data/deita_sota_pool'
    knowledge_data_file = f'../data/deita_sota_pool/model_metric/{model_name}_deita_sota_pool_305263_token_4096.json'
    knowledge_data = _load_data(knowledge_data_file)


    complexity_quality_knowledge_data = []
    add_keys = list(knowledge_data[0][0][0].keys())
    print(f'add_keys:{add_keys}')
    keys_map = {
        'ppl' : 'ppl',
        'loss' : 'loss',
        'Min_5.0% Prob' : 'knowledge5_scores',
        'Min_10.0% Prob' : 'knowledge10_scores',
        'Min_20.0% Prob' : 'knowledge20_scores',
        'Min_30.0% Prob' : 'knowledge30_scores',
        'Min_40.0% Prob' : 'knowledge40_scores',
        'Min_50.0% Prob' : 'knowledge50_scores',
        'Min_60.0% Prob' : 'knowledge60_scores',
    }

    for index_i, item in enumerate(add_complexity_quality_scores):

        for add_key in add_keys:
            item[keys_map[add_key]] = []
            for q_a_score in knowledge_data[index_i]:
                if len(q_a_score) == 2:
                    item[keys_map[add_key]].append(q_a_score[0][add_key] + q_a_score[1][add_key])
        complexity_quality_knowledge_data.append(item)

    output_path = f'{data_dir}/model_metric'
    output_file = 'qck_' + model_name + '_' + data_name + '.json'


    # print(f'{output_path}/_stage_{optimize_stage}_{rank_model}/{output_file}')
    # with open(f'{output_path}/_stage_{optimize_stage}_{rank_model}/{output_file}', "w") as json_file:
    #     json.dump(complexity_quality_knowledge_data, json_file)

    print(f'{output_path}/_stage_c4q3_{rank_model}/{output_file}')
    with open(f'{output_path}/_stage_c4q3_{rank_model}/{output_file}', "w") as json_file:
        json.dump(complexity_quality_knowledge_data, json_file)



add_keys:['ppl', 'loss', 'Min_5.0% Prob', 'Min_10.0% Prob', 'Min_20.0% Prob', 'Min_30.0% Prob', 'Min_40.0% Prob', 'Min_50.0% Prob', 'Min_60.0% Prob']
../data/deita_sota_pool/model_metric/_stage_c4q3_deberta-v3-base/qck_llama2-13b-hf_deita_sota_pool_305263_token_4096.json
add_keys:['ppl', 'loss', 'Min_5.0% Prob', 'Min_10.0% Prob', 'Min_20.0% Prob', 'Min_30.0% Prob', 'Min_40.0% Prob', 'Min_50.0% Prob', 'Min_60.0% Prob']
../data/deita_sota_pool/model_metric/_stage_c4q3_deberta-v3-base/qck_Mistral-7B-v0.1_deita_sota_pool_305263_token_4096.json
add_keys:['ppl', 'loss', 'Min_5.0% Prob', 'Min_10.0% Prob', 'Min_20.0% Prob', 'Min_30.0% Prob', 'Min_40.0% Prob', 'Min_50.0% Prob', 'Min_60.0% Prob']
../data/deita_sota_pool/model_metric/_stage_c4q3_deberta-v3-base/qck_llama-13b-hf_deita_sota_pool_305263_token_4096.json


In [21]:
c3_q2_data1=_load_data('../data/deita_sota_pool/model_metric/_stage_c3q2_deberta-v3-large/qck_llama-13b-hf_deita_sota_pool_305263_token_4096.json')


c3_data    = _load_data('../data/deita_sota_pool/model_metric/_stage_3_deberta-v3-large/qck_llama-13b-hf_deita_sota_pool_305263_token_4096.json')

q2_data    = _load_data('../data/deita_sota_pool/model_metric/_stage_2_deberta-v3-large/qck_llama-13b-hf_deita_sota_pool_305263_token_4096.json')

In [23]:
print(c3_q2_data1[100]['complexity_scores'])
print(c3_q2_data1[100]['quality_scores'])

print(c3_data[100]['complexity_scores'])
print(q2_data[100]['quality_scores'])

[-4.26171875, -5.34375, -3.39453125, -4.82421875, -1.203125, -2.8046875]
[2.177734375, 1.48046875, 3.52734375, 2.173828125, 4.22265625, 3.40234375]
[-4.26171875, -5.34375, -3.39453125, -4.82421875, -1.203125, -2.8046875]
[2.177734375, 1.48046875, 3.52734375, 2.173828125, 4.22265625, 3.40234375]


In [ ]:

import os

stage_list = [optimize_stage] # [0,1,2,3,4,5]
data_size = 10000
knowledge=20
embedding_model = 'sentence-transformers-e5-large-v2'
model_name_list = ['Mistral-7B-v0.1', 'llama-13b-hf', 'llama2-13b-hf'] # , 'llama-7b-hf' 'llama2-7b-hf', 

for optimize_stage in stage_list:
    for model_name in model_name_list:
        data_dir = '../data/deita_sota_pool'
        output_file = f'_qck_sentence-transformers-e5-large-v2_' + model_name + f'_{rank_model}_' + f'deita_sota_pool_305263_{data_size}_0.9_{knowledge}.json'
        # tmp_output_file = f'_qck_stage_{optimize_stage}_sentence-transformers-e5-large-v2_' + model_name + f'_{rank_model}_' + f'deita_sota_pool_305263_{data_size}_0.9_{knowledge}.json'
        tmp_output_file = f'_qck_stage_c3q2_sentence-transformers-e5-large-v2_' + model_name + f'_{rank_model}_' + f'deita_sota_pool_305263_{data_size}_0.9_{knowledge}.json'



        # output_path = f'{data_dir}/model_metric/_stage_{optimize_stage}_{rank_model}'
        output_path = f'{data_dir}/model_metric/_stage_c3q2_deberta-v3-large'
        cmd1 = f'cp {output_path}/{output_file}   {output_path}/{tmp_output_file}'
        cmd2 = f'mv {output_path}/{tmp_output_file}  ../LLaMA-Factory/data/'
        print(cmd1)
        print(cmd2)
        os.system(cmd1)
        os.system(cmd2)

In [16]:
'''
Distillation Refine
'''

def random_selection_indices(data_list, num_bins=10, num_samples=3):

    counts, bins = np.histogram(data_list, bins=num_bins)

    bin_edges = bins.tolist()

    bin_indices = [[] for _ in range(num_bins)]

    for i, num in enumerate(data_list):
        for j in range(len(bin_edges) - 1):
            if bin_edges[j] <= num < bin_edges[j + 1]:
                bin_indices[j].append(i)
                break

    selected_indices = []
    for indices in bin_indices:
        if len(indices) >= num_samples:
            selected_indices.append(np.random.choice(indices, size=num_samples, replace=False).tolist())
        else:
            selected_indices.append(np.random.choice(indices, size=num_samples, replace=True).tolist())
            # selected_indices.append(indices)

    return selected_indices, counts, bin_edges

# random_selection_indices(data_list=[i for i in range(10)] , num_bins=10, num_samples=3)

complexity_quality_data = _load_data(f'{output_path}/qck_llama-13b-hf_deita_sota_pool_305263_token_4096.json')
print(f'{output_path}/qck_llama-13b-hf_deita_sota_pool_305263_token_4096.json')
complexity_bins = []
quality_bins    = []
for index,item in enumerate(complexity_quality_data):
    complexity_bins.append(item['complexity_scores'][0])
    quality_bins.append(item['quality_scores'][0])

print(len(quality_bins))


../data/deita_sota_pool/model_metric/_stage_4_deberta-v3-large/qck_llama-13b-hf_deita_sota_pool_305263_token_4096.json
305263


In [17]:

np.random.seed(42)
quality_check_indices, quality_counts, quality_bin_edges = random_selection_indices(data_list=quality_bins, num_bins=5, num_samples=1000)
complexity_check_indices, complexity_counts, complexity_bin_edges = random_selection_indices(data_list=complexity_bins, num_bins=5, num_samples=1000)

print(f'max(quality_bins):{max(quality_bins)}, min(quality_bins):{min(quality_bins)},\nmax(complexity_bins):{max(complexity_bins)}, min(complexity_bins):{min(complexity_bins)}')

rank_quality_base_instruction = """Rank the following QUESTION and RESPONSE pairs based on their relevance to the QUESTION.
Your evaluation should consider factors such as helpfulness, relevance, accuracy, depth, creativity, and level of detail of the response.
The following are {num} QUESTION and RESPONSE pairs, each indicated by number identifier [].
You can rank them based on their relevance to the QUESTION:

{question_response_pairs}

The responses will be listed in descending order using identifiers, and the most relevant response should be listed first, and the output format should be [] > [] > etc, e.g., [1] > [2] > etc.
The ranking results of the {num} QUESTION and RESPONSE pairs (only identifiers) is:
"""


def createRankQualityPrompt(question_response_pairs, num, chat_mode=False):
    if not chat_mode:
        
        prompt = rank_quality_base_instruction.format(question_response_pairs=question_response_pairs, num=num)
        
        history=[{"role": "system", "content": "You are RankGPT, an intelligent assistant that can rank responses based on their questions."}, {"role": "user", "content": prompt}]
    
    else:
        history=[
            {"role": "system", "content": "You are RankGPT, an intelligent assistant that can rank responses based on their questions."},
            {"role": "user", "content": "I will provide you with {num} responses, each indicated by number identifier []. Rank them based on their relevance to question: {question}.\nYour evaluation should consider factors such as helpfulness, relevance, accuracy, depth, creativity, and level of detail of the response.".format(num, question)},
            {"role": "assistant", "content": "Okay, please provide the responses."}
        ]

        user_assistant_history = []
        for index,item in enumerate(responses):
            user_assistant_history.append(
               {"role": "user", "content": item}
            )
            user_assistant_history.append(
               {"role": "assistant", "content": "Received responses [{}]".format(index+1)}
            )
        history = history + user_assistant_history + [{
            "role": "assistant", "content": "Search question: {question}.\nRank the {num} responses above based on their relevance to the search question. The responses should be listed in descending order using identifiers, and the most relevant response should be listed first, and the output format should be [] > [], e.g., [1] > [2]. Only return the ranking results, do not say any word or explain.".format(question=question, num=num)
        }]
        
    return history, prompt




rank_complexity_base_instruction = """Ranking the following instructions provided by different users according to the instruction-following difficulty and complexity.
The following are {num} instructions, each indicated by number identifier [].
The user's instructions are: 

{instructions}

You will rank the {num} instructions above based on the instruction-following difficulty and complexity.
The instructions will be listed in descending order using identifiers, and the most relevant response should be listed first, and the output format should be [] > [] > etc, e.g., [1] > [2] > etc.
The ranking results of the {num} responses (only identifiers) is:
"""

def createRankComplexityPrompt(instructions, num, chat_mode=False):
    if not chat_mode:
        
        prompt = rank_complexity_base_instruction.format(instructions=instructions, num=num)
        
        history=[{"role": "system", "content": "You are RankGPT, an intelligent assistant that can rank instructions based on their instruction-following difficulty and complexity."}, {"role": "user", "content": prompt}]
    
    else:
        history=[
            {"role": "system", "content": "You are RankGPT, an intelligent assistant that can rank instructions based on the instruction-following difficulty and complexity."},
            {"role": "user", "content": "I will provide you with {num} instructions, each indicated by number identifier []. Rank them based on their instruction-following difficulty and complexity.".format(num)},
            {"role": "assistant", "content": "Okay, please provide the instructions."}
        ]

        user_assistant_history = []
        for index,item in enumerate(instructions):
            user_assistant_history.append(
               {"role": "user", "content": item}
            )
            user_assistant_history.append(
               {"role": "assistant", "content": "Received instructions [{}]".format(index+1)}
            )
        history = history + user_assistant_history + [{
            "role": "assistant", "content": "Rank the {num} instructions above based on their instruction-following difficulty and complexity. The instructions should be listed in descending order using identifiers, and the most complex and difficult instruction should be listed first, and the output format should be [] > [], e.g., [1] > [2]. Only return the ranking results, do not say any word or explain.".format(num=num)
        }]
        
    return history, prompt


quality_check_indices = np.array(quality_check_indices).T
complexity_check_indices = np.array(complexity_check_indices).T

quality_history_list = []
quality_scores_list = []
quality_prompt_list = []
for index_list in quality_check_indices:
    q_a_list = []
    scores = []
    for i,index in enumerate(index_list):
        q = add_complexity_quality_scores[index]['conversations'][0]['value'] # q
        a = add_complexity_quality_scores[index]['conversations'][1]['value'] # a
        scores.append(add_complexity_quality_scores[index]['quality_scores'][0])
        q_a_list.append('['+str(int(i+1))+'] ' + f'QUESTION: {q}\n\nRESPONSE: {a}')
    question_response_pairs='\n'.join(q_a_list)
    history, prompt = createRankQualityPrompt(question_response_pairs=question_response_pairs, num=len(q_a_list))
    quality_prompt_list.append(prompt)
    quality_history_list.append(history)
    quality_scores_list.append(scores)

complexity_history_list = []
complexity_scores_list = []
complexity_prompt_list = []
for index_list in complexity_check_indices:
    q_list = []
    scores = []
    for i,index in enumerate(index_list):
        q = add_complexity_quality_scores[index]['conversations'][0]['value'] # q
        scores.append(add_complexity_quality_scores[index]['complexity_scores'][0])
        q_list.append('['+str(int(i+1))+'] ' + f'{q}')
    instructions='\n'.join(q_list)
    history, prompt = createRankComplexityPrompt(instructions=instructions, num=len(q_list))
    complexity_prompt_list.append(prompt)
    complexity_history_list.append(history)
    complexity_scores_list.append(scores)

# sorted(enumerate([1,2,3,4]), key=lambda x: x[1], reverse=True)
# [(3, 4), (2, 3), (1, 2), (0, 1)] (index, value)

def score_model_rank(scores):
    print(f'scores:{scores}')
    sorted_indexes = [idx+1 for idx, _ in sorted(enumerate(scores), key=lambda x: x[1], reverse=True)]
    print(f'sorted_indexes:{sorted_indexes}')

max(quality_bins):2.14453125, min(quality_bins):-10.6328125,
max(complexity_bins):9.9609375, min(complexity_bins):-11.0625


In [18]:
complexity_bin_edges

[-11.0625,
 -6.8578125,
 -2.6531249999999993,
 1.551562500000001,
 5.756250000000001,
 9.9609375]

In [19]:
test_index = 0
print(complexity_prompt_list[test_index])
score_model_rank(complexity_scores_list[test_index])

Ranking the following instructions provided by different users according to the instruction-following difficulty and complexity.
The following are 5 instructions, each indicated by number identifier [].
The user's instructions are: 

[1] Generate an engaging line of dialogue that your character would say in a movie.
[2] Juan had a total of 18 matches and is planning to buy two matches. How many matches can Juan choose to buy?
[3] Develop an informative and engaging tutorial on the advanced features of Adobe Illustrator that are particularly useful for creating realistic digital portraits, while also highlighting the lesser-known tools and techniques that can help take these portraits to the next level.
[4] How can we implement machine-to-machine communication and collaboration in a self-sustaining manner using C# while also ensuring that the machines maintain their autonomy and decision-making capabilities? Can you provide a detailed scenario where two or more machines with different f

In [20]:
test_index = 10
print(quality_prompt_list[test_index])
score_model_rank(quality_scores_list[test_index])


# Spearman’s Rank Correlation 

Rank the following QUESTION and RESPONSE pairs based on their relevance to the QUESTION.
Your evaluation should consider factors such as helpfulness, relevance, accuracy, depth, creativity, and level of detail of the response.
The following are 5 QUESTION and RESPONSE pairs, each indicated by number identifier [].
You can rank them based on their relevance to the QUESTION:

[1] QUESTION: How did the show's themes and subject matter relate to the cultural and political climate of the time it was produced?

RESPONSE: I do not have any knowledge of specific tv shows unless provided with more information. please provide me with the name of the show you are referring to, and i will be happy to answer your question.
[2] QUESTION: What’s happening in Single Wall Corrugated Tube Sales Market 2018?
Global Single Wall Corrugated Tube Sales Market report provides a detailed analysis of the market from 2013 to 2023. The report provides comprehensive analysis for the US, Canada, Japan, Europe, Asia

In [21]:
def process_rank_str(rank_str: str):

    def clean_response(response: str):
        pattern = r'\[\d+\]'
        matches = re.findall(pattern, response)
        response = ' > '.join(matches)
        new_response = ''
        for c in response:
            if not c.isdigit():
                new_response += ' '
            else:
                new_response += c
        new_response = new_response.strip()
        return new_response

    def remove_duplicate(response):
        new_response = []
        for c in response:
            if c not in new_response:
                new_response.append(c)
        return new_response
    
    rank_str = clean_response(rank_str)
    rank_list = [int(x) for x in rank_str.split()]
    rank_list = remove_duplicate(rank_list)

    return rank_list

s='[1] > [18] > [14] > [1] > [15] > [3] > [4] > [13]'
new_s = process_rank_str(s)
print(new_s)

[1, 18, 14, 15, 3, 4, 13]


In [22]:
def get_oai_completion(history):

    try: 
        response = openai.ChatCompletion.create(
          model="gpt-3.5-turbo",
          messages=history,
           temperature=1,
           max_tokens=2048,
           top_p=0.95,
           frequency_penalty=0,
           presence_penalty=0,
           stop=None
        )
        res = response["choices"][0]["message"]["content"]
       
        gpt_output = res
        return gpt_output
    except requests.exceptions.Timeout:
        # Handle the timeout error here
        print("The OpenAI API request timed out. Please try again later.")
        return None
    except openai.error.InvalidRequestError as e:
        # Handle the invalid request error here
        print(f"The OpenAI API request was invalid: {e}")
        return None
    except openai.error.APIError as e:
        if "The operation was timeout" in str(e):
            # Handle the timeout error here
            print("The OpenAI API request timed out. Please try again later.")
#             time.sleep(3)
            return get_oai_completion(history)            
        else:
            # Handle other API errors here
            print(f"The OpenAI API returned an error: {e}")
            return None
    except openai.error.RateLimitError as e:
        return get_oai_completion(history)

def call_chatgpt(history):
    success = False
    re_try_count = 10
    ans = ''


    while not success and re_try_count >= 0:
        re_try_count -= 1
        try:
            ans = get_oai_completion(history)
            if ans:
                success = True
        except:
            time.sleep(15)
            print('retry for sample:', history)
    return ans

In [23]:
len(complexity_history_list)

1000

In [24]:
f'../data/RankGPT/_complexity_stage_{optimize_stage}_{rank_model}_feedback_{data_name}.json'

'../data/RankGPT/_complexity_stage_4_deberta-v3-large_feedback_deita_sota_pool_305263_token_4096.json'

In [25]:
output_file_path = f'../data/RankGPT/_complexity_stage_{optimize_stage}_{rank_model}_feedback_{data_name}.json'

complexity_history_answer_list = []
for index,history in tqdm(enumerate(complexity_history_list)):
    try:
        answer = call_chatgpt(history) # '[3] > [5] > [4] > [2] > [1]'
    except:
        time.sleep(10)
        answer = call_chatgpt(history) # '[3] > [5] > [4] > [2] > [1]'
    complexity_history_answer_list.append(
        {
            'teacher_answer': answer,
            'history': history,
            'data_index': complexity_check_indices[index].tolist(),
            'student_answer': [i+1 for i in range(len(complexity_scores_list[index]))][::-1]
         }
    )


with open(output_file_path, 'w') as f:
    json.dump(complexity_history_answer_list, f, indent=4)


0it [00:00, ?it/s]

1000it [21:45,  1.31s/it]


In [26]:
f'../data/RankGPT/_complexity_stage_{optimize_stage}_{rank_model}_feedback_{data_name}.json'

'../data/RankGPT/_complexity_stage_4_deberta-v3-large_feedback_deita_sota_pool_305263_token_4096.json'

In [27]:
output_file_path = f'../data/RankGPT/_complexity_stage_{optimize_stage}_{rank_model}_feedback_{data_name}.json'
complexity_history_answer_list = _load_data(output_file_path)
p_value_threshold = 0.05 # less is better

rho_value_threshold = 0.8 # large is better
complexity_bad_list = []
for index,item in enumerate(complexity_history_answer_list):
    try:
        ytrue = process_rank_str(item['teacher_answer'])
    except:
        print(index)
        print(item)
    ypredict = item['student_answer']
    try:
        rho, p_value = calc_spearmanr(ytrue, ypredict)
    except:
        print(index)
        print(item)
    if p_value > p_value_threshold:
        complexity_bad_list.append(index)
        # print(f"Spearman's Rank Correlation: {rho:.4f}, p-value: {p_value:.4f}")

    # if rho < rho_value_threshold:
    #     complexity_bad_list.append(index)
    #     print(f"Spearman's Rank Correlation: {rho:.4f}, p-value: {p_value:.4f}")

print(f'complexity_bad_list: {len(complexity_bad_list)}')

complexity_bad_list: 243


In [65]:
# call_chatgpt(history)


call_chatgpt(complexity_history_answer_list[275]['history'])

'[5] > [3] > [2] > [1]'

In [28]:
print(complexity_history_answer_list[275]['history'][1]['content'])

Ranking the following instructions provided by different users according to the instruction-following difficulty and complexity.
The following are 5 instructions, each indicated by number identifier [].
The user's instructions are: 

[1] Can you recommend any places for fishing in Juneau, Alaska during June?
[2] Write a job description for a software engineer.
[3] Romulo Chaidez Medrano and Arcenio Chaidez Medrano are charged with more than 14-counts of drug related offenses.
Two Mexican nationals have been indicted in a $3 million drug bust at a Hasbrouck Heights hotel in November.
Romulo Chaidez Medrano and Arcenio Chaidez Medrano, who are brothers, were part of a surveillance sting at the Hilton Hotel on Terrance Avenue, where police allegedly uncovered the drug stash in their possession. In all, authorities said they seized 13 kilograms of crystal methamphetamine, 3½ kilograms of heroin, 6,250 tabs of fentanyl and 3 pounds of raw fentanyl, a potent synthetic opioid.
A grand jury on

In [29]:
complexity_history_answer_list[303]['history']

[{'role': 'system',
  'content': 'You are RankGPT, an intelligent assistant that can rank instructions based on their instruction-following difficulty and complexity.'},
 {'role': 'user',
  'content': "Ranking the following instructions provided by different users according to the instruction-following difficulty and complexity.\nThe following are 5 instructions, each indicated by number identifier [].\nThe user's instructions are: \n\n[1] What is the most popular tourist attraction in Bali?\n[2] Compare and contrast articles of confederation and the Constitution.\n[3] Please comment the business plan and state the improvement to this business plan: \n\nCompany Name: Fintech F&B Solutions\n\nExecutive Summary\n\nFintech F&B Solutions aims to revolutionize the food and beverage (F&B) industry by offering a comprehensive digital platform that streamlines supply chain transactions and provides tailored credit line solutions for restaurants. By partnering with major food distributors like 

In [30]:
f'../data/RankGPT/_quality_stage_{optimize_stage}_{rank_model}_feedback_{data_name}.json'

'../data/RankGPT/_quality_stage_4_deberta-v3-large_feedback_deita_sota_pool_305263_token_4096.json'

In [32]:
optimize_stage

4

In [33]:
output_file_path = f'../data/RankGPT/_quality_stage_{optimize_stage}_{rank_model}_feedback_{data_name}.json'
#                  f'../data/RankGPT/_complexity_stage_{optimize_stage}_{rank_model}_feedback_{data_name}.json'
quality_history_answer_list = []
for index,history in tqdm(enumerate(quality_history_list)):
    try:
        answer = call_chatgpt(history) # '[3] > [5] > [4] > [2] > [1]'
    except:
        time.sleep(10)
        answer = call_chatgpt(history) # '[3] > [5] > [4] > [2] > [1]'
    quality_history_answer_list.append(
        {
            'teacher_answer': answer,
            'history': history,
            'data_index': quality_check_indices[index].tolist(),
            'student_answer': [i+1 for i in range(len(quality_scores_list[index]))][::-1]
         }
    )

with open(output_file_path, 'w') as f:
    json.dump(quality_history_answer_list, f, indent=4)

1000it [22:05,  1.33s/it]


In [34]:
quality_history_answer_list[104]['history']

[{'role': 'system',
  'content': 'You are RankGPT, an intelligent assistant that can rank responses based on their questions.'},
 {'role': 'user',
  'content': 'Rank the following QUESTION and RESPONSE pairs based on their relevance to the QUESTION.\nYour evaluation should consider factors such as helpfulness, relevance, accuracy, depth, creativity, and level of detail of the response.\nThe following are 5 QUESTION and RESPONSE pairs, each indicated by number identifier [].\nYou can rank them based on their relevance to the QUESTION:\n\n[1] QUESTION: Hello, human. Being an advanced AI agent, my goal is to be a part of the human society without being detected by studying human behavior patterns. Could you please provide me with a detailed account of your routine to assist me in understanding human thought processes better? Please specify all intricate decision-making procedures that you undertake on a daily basis, including any environmental factors, social interactions, or cognitive ch

In [35]:
f'../data/RankGPT/_quality_stage_{optimize_stage}_{rank_model}_feedback_{data_name}.json'

'../data/RankGPT/_quality_stage_4_deberta-v3-large_feedback_deita_sota_pool_305263_token_4096.json'

In [53]:
quality_history_answer_list = _load_data(f'../data/RankGPT/_quality_stage_{optimize_stage}_{rank_model}_feedback_{data_name}.json')

p_value_threshold = 0.05 # less is better
rho_value_threshold = 0.8 # large is better
quality_bad_list = []
for index,item in enumerate(quality_history_answer_list):
    try:
        ytrue = process_rank_str(item['teacher_answer'])
    except:
        print(f'process error index:{index}')
        print(item)
    ypredict = item['student_answer']

    try:
        rho, p_value = calc_spearmanr(ytrue, ypredict)
    except:
        print(f'calc_spearmanr error index:{index}')
        print(index)
    if p_value > p_value_threshold:
        quality_bad_list.append(index)
        # print(f"Spearman's Rank Correlation: {rho:.4f}, p-value: {p_value:.4f}")

    # if rho < rho_value_threshold:
    #     quality_bad_list.append(index)
    #     print(f"Spearman's Rank Correlation: {rho:.4f}, p-value: {p_value:.4f}")

print(f'len(quality_bad_list):{len(quality_bad_list)}')

len(quality_bad_list):321


In [49]:
quality_history_answer_list[703]

{'teacher_answer': '[5] > [4] > [2] > [1]',
 'history': [{'role': 'system',
   'content': 'You are RankGPT, an intelligent assistant that can rank responses based on their questions.'},
  {'role': 'user',
   'content': 'Rank the following QUESTION and RESPONSE pairs based on their relevance to the QUESTION.\nYour evaluation should consider factors such as helpfulness, relevance, accuracy, depth, creativity, and level of detail of the response.\nThe following are 5 QUESTION and RESPONSE pairs, each indicated by number identifier [].\nYou can rank them based on their relevance to the QUESTION:\n\n[1] QUESTION: Is there a gift shop located within the museum? If so, what kinds of items can be purchased there?\n\nRESPONSE: I am sorry, but as an AI language model, I do not have information regarding a specific museum\'s gift shop. Please provide more details about the museum you are referring to.\n[2] QUESTION: What types of art and crafts are typically sold at Halifax\'s cultural events and

In [51]:
call_chatgpt(quality_history_answer_list[703]['history'])

'[5] > [4] > [2] > [1]'

In [52]:
print(quality_history_answer_list[703]['history'][1]['content'])

Rank the following QUESTION and RESPONSE pairs based on their relevance to the QUESTION.
Your evaluation should consider factors such as helpfulness, relevance, accuracy, depth, creativity, and level of detail of the response.
The following are 5 QUESTION and RESPONSE pairs, each indicated by number identifier [].
You can rank them based on their relevance to the QUESTION:

[1] QUESTION: Is there a gift shop located within the museum? If so, what kinds of items can be purchased there?

RESPONSE: I am sorry, but as an AI language model, I do not have information regarding a specific museum's gift shop. Please provide more details about the museum you are referring to.
[2] QUESTION: What types of art and crafts are typically sold at Halifax's cultural events and where can they be found?

RESPONSE: As an AI language model, I don't have access to up-to-date information on cultural events in Halifax or their product offerings. However, typical art and crafts sold at cultural events in Halif

In [ ]:
len(complexity_bad_list)

218

In [54]:
'''
complexity refine
'''
orig_stage = "_stage_4_deberta-v3-large"
next_stage = '_stage_5_deberta-v3-large'
orig_complexity_data_path = f'../data/RankGPT/{orig_stage}/alpaca-train-complexity-2k.json'
orig_complexity_data_rank_path = f'../data/RankGPT/{orig_stage}/alpaca-train-complexity-rank-2k.json'
mix_data_path = f'../data/RankGPT/{next_stage}/alpaca-train-complexity-2k.json'
mix_data_rank_path = f'../data/RankGPT/{next_stage}/alpaca-train-complexity-rank-2k.json'


format_complexity_feedback_for_train = []
format_complexity_feedback_for_train_rank = []
orig_complexity_data = _load_data(orig_complexity_data_path)
orig_complexity_data_rank = _load_data(orig_complexity_data_rank_path)

for item_i in complexity_bad_list:
    format_complexity_feedback_for_train_rank.append(complexity_history_answer_list[item_i]['teacher_answer'])


    # dict_keys(['teacher_answer', 'history', 'data_index', 'student_answer'])
    select_list = complexity_history_answer_list[item_i]['data_index']
    tmp_dict = { }
    query = []
    query_id= ""
    positive_passages= []
    retrieved_passages = []
    for item_j in select_list:
        query.append(data[item_j]['conversations'][0]['value'])  # 0 human

        retrieved_passages.append(
            {
                "instruction" : data[item_j]['conversations'][0]['value'],
                "input": "",
                "output": data[item_j]['conversations'][1]['value'], # 1 gpt
                "index": item_j,
                "text" : data[item_j]['conversations'][1]['value'] # 1 gpt
            }
        )
    tmp_dict["query"] = query
    tmp_dict["query_id"] = query_id
    tmp_dict['positive_passages'] = positive_passages
    tmp_dict['retrieved_passages'] = retrieved_passages
    format_complexity_feedback_for_train.append(tmp_dict)


mix_data = orig_complexity_data + format_complexity_feedback_for_train
mix_rank = orig_complexity_data_rank + format_complexity_feedback_for_train_rank

with open(mix_data_path, "w") as json_file:
    json.dump(mix_data, json_file, indent=4)

with open(mix_data_rank_path, "w") as json_file:
    json.dump(mix_rank, json_file, indent=4)

In [55]:
'''
quality refine
'''
orig_stage = "_stage_4_deberta-v3-large"
next_stage = '_stage_5_deberta-v3-large'
orig_quality_data_path = f'../data/RankGPT/{orig_stage}/alpaca-train-quality-2k.json'
orig_quality_data_rank_path = f'../data/RankGPT/{orig_stage}/alpaca-train-quality-rank-2k.json'
mix_data_path = f'../data/RankGPT/{next_stage}/alpaca-train-quality-2k.json'
mix_data_rank_path = f'../data/RankGPT/{next_stage}/alpaca-train-quality-rank-2k.json'


format_quality_feedback_for_train = []
format_quality_feedback_for_train_rank = []
orig_quality_data = _load_data(orig_quality_data_path)
orig_quality_data_rank = _load_data(orig_quality_data_rank_path)

for item_i in quality_bad_list:
    format_quality_feedback_for_train_rank.append(quality_history_answer_list[item_i]['teacher_answer'])


    # dict_keys(['teacher_answer', 'history', 'data_index', 'student_answer'])
    select_list = quality_history_answer_list[item_i]['data_index']
    tmp_dict = { }
    query = []
    query_id= ""
    positive_passages= []
    retrieved_passages = []
    for item_j in select_list:
        query.append(data[item_j]['conversations'][0]['value'])  # 0 human

        retrieved_passages.append(
            {
                "instruction" : data[item_j]['conversations'][0]['value'],
                "input": "",
                "output": data[item_j]['conversations'][1]['value'], # 1 gpt
                "index": item_j,
                "text" : data[item_j]['conversations'][1]['value'] # 1 gpt
            }
        )
    tmp_dict["query"] = query
    tmp_dict["query_id"] = query_id
    tmp_dict['positive_passages'] = positive_passages
    tmp_dict['retrieved_passages'] = retrieved_passages
    format_quality_feedback_for_train.append(tmp_dict)


mix_data = orig_quality_data + format_quality_feedback_for_train
mix_rank = orig_quality_data_rank + format_quality_feedback_for_train_rank

with open(mix_data_path, "w") as json_file:
    json.dump(mix_data, json_file, indent=4)

with open(mix_data_rank_path, "w") as json_file:
    json.dump(mix_rank, json_file, indent=4)

In [25]:
len(mix_data)

3416

In [56]:
len(quality_history_answer_list)

999